# inferecne for embeddings generation with using trained CodeGraphNet archetchture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import RobertaTokenizer, RobertaModel
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings
import ast
warnings.filterwarnings('ignore')

import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"


class ICFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_icfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            func_calls = {}
            func_defs = {}
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, ast.FunctionDef):
                    func_defs[node.name] = current_id
                
                if isinstance(node, ast.Call):
                    if isinstance(node.func, ast.Name):
                        func_name = node.func.id
                        if func_name not in func_calls:
                            func_calls[func_name] = []
                        func_calls[func_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    traverse(child, current_id)
            
            traverse(tree)
            
            for func_name, call_sites in func_calls.items():
                if func_name in func_defs:
                    def_id = func_defs[func_name]
                    for call_id in call_sites:
                        edges.append([call_id, def_id])
                        edges.append([def_id, call_id])
            
            if len(nodes) == 0:
                nodes = ['Module']
                edges = []
            
            return nodes, edges
        except:
            return ['Module'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class DFGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_dfg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            var_last_write = {}
            var_last_read = {}
            
            def extract_dataflow(node, current_id):
                if isinstance(node, ast.Name):
                    var_name = node.id
                    
                    if isinstance(node.ctx, ast.Store):
                        if var_name in var_last_read:
                            for read_id in var_last_read[var_name]:
                                edges.append([read_id, current_id])
                        var_last_write[var_name] = current_id
                        var_last_read[var_name] = []
                    
                    elif isinstance(node.ctx, ast.Load):
                        if var_name in var_last_write:
                            edges.append([var_last_write[var_name], current_id])
                        if var_name not in var_last_read:
                            var_last_read[var_name] = []
                        var_last_read[var_name].append(current_id)
                
                for child in ast.iter_child_nodes(node):
                    extract_dataflow(child, current_id)
            
            def traverse(node):
                nonlocal node_id
                current_id = node_id
                nodes.append(type(node).__name__)
                node_id += 1
                extract_dataflow(node, current_id)
                for child in ast.iter_child_nodes(node):
                    traverse(child)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class CDGBuilder:
    def __init__(self, graphcodebert_model):
        self.graphcodebert = graphcodebert_model
        
    def build_cdg(self, code_snippet):
        try:
            tree = ast.parse(code_snippet)
            nodes = []
            edges = []
            node_id = 0
            control_stack = []
            
            def traverse(node, parent_id=None):
                nonlocal node_id
                current_id = node_id
                node_type = type(node).__name__
                nodes.append(node_type)
                node_id += 1
                
                if parent_id is not None:
                    edges.append([parent_id, current_id])
                
                if isinstance(node, (ast.If, ast.For, ast.While, ast.Try)):
                    for ctrl_id in control_stack:
                        edges.append([ctrl_id, current_id])
                    control_stack.append(current_id)
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
                    
                    control_stack.pop()
                else:
                    if control_stack:
                        for ctrl_id in control_stack:
                            edges.append([ctrl_id, current_id])
                    
                    for child in ast.iter_child_nodes(node):
                        traverse(child, current_id)
            
            traverse(tree)
            
            if len(nodes) == 0:
                nodes = ['Program']
                edges = []
            
            return nodes, edges
        except:
            return ['Program'], []
    
    def nodes_to_embeddings(self, nodes, tokenizer, device):
        embeddings = []
        for node_type in nodes:
            tokens = tokenizer(node_type, return_tensors='pt', padding=True, truncation=True, max_length=16)
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                outputs = self.graphcodebert(**tokens)
                embedding = outputs.last_hidden_state.mean(dim=1).squeeze(0)
            embeddings.append(embedding)
        return torch.stack(embeddings)


class StableGraphEncoder(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=384, output_dim=512):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(0.2)
        self.residual = nn.Linear(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        identity = self.residual(x)
        x = self.conv1(x, edge_index)
        x = self.norm1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.norm2(x)
        x = x + identity
        return x


class HierarchicalGraphFusion(nn.Module):
    def __init__(self, graph_dim=512, num_heads=8):
        super().__init__()
        self.graph_dim = graph_dim
        self.icfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.dfg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.cdg_encoder = StableGraphEncoder(768, 384, graph_dim)
        self.graph_attention = nn.MultiheadAttention(graph_dim, num_heads, dropout=0.1, batch_first=True)
        self.graph_norm = nn.LayerNorm(graph_dim)
        self.gate = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim),
            nn.Sigmoid()
        )
        self.fusion = nn.Sequential(
            nn.Linear(graph_dim * 3, graph_dim * 2),
            nn.LayerNorm(graph_dim * 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(graph_dim * 2, graph_dim)
        )
        
    def forward(self, icfg_data, dfg_data, cdg_data):
        h_icfg = self.icfg_encoder(icfg_data.x, icfg_data.edge_index)
        h_dfg = self.dfg_encoder(dfg_data.x, dfg_data.edge_index)
        h_cdg = self.cdg_encoder(cdg_data.x, cdg_data.edge_index)
        icfg_global = torch.mean(h_icfg, dim=0, keepdim=True)
        dfg_global = torch.mean(h_dfg, dim=0, keepdim=True)
        cdg_global = torch.mean(h_cdg, dim=0, keepdim=True)
        graph_stack = torch.stack([icfg_global, dfg_global, cdg_global], dim=1)
        attended, _ = self.graph_attention(graph_stack, graph_stack, graph_stack)
        attended = self.graph_norm(attended + graph_stack)
        fused = torch.cat([attended[:, 0], attended[:, 1], attended[:, 2]], dim=-1)
        gate_weights = self.gate(fused)
        output = self.fusion(fused)
        output = output * gate_weights
        return output.squeeze(0)


class EmbeddingExtractor(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.tokenizer = RobertaTokenizer.from_pretrained('microsoft/graphcodebert-base')
        self.graphcodebert = RobertaModel.from_pretrained('microsoft/graphcodebert-base')
        
        for param in self.graphcodebert.parameters():
            param.requires_grad = False
        
        self.icfg_builder = ICFGBuilder(self.graphcodebert)
        self.dfg_builder = DFGBuilder(self.graphcodebert)
        self.cdg_builder = CDGBuilder(self.graphcodebert)
        self.graph_fusion = HierarchicalGraphFusion(graph_dim=512, num_heads=8)
        self.code_projection = nn.Linear(768, 512)
        self.multimodal_fusion = nn.Sequential(
            nn.Linear(512 + 512, 768),
            nn.LayerNorm(768),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(768, 512),
            nn.LayerNorm(512)
        )
        
    def build_graph_data(self, code, device):
        icfg_nodes, icfg_edges = self.icfg_builder.build_icfg(code)
        dfg_nodes, dfg_edges = self.dfg_builder.build_dfg(code)
        cdg_nodes, cdg_edges = self.cdg_builder.build_cdg(code)
        
        icfg_x = self.icfg_builder.nodes_to_embeddings(icfg_nodes, self.tokenizer, device)
        dfg_x = self.dfg_builder.nodes_to_embeddings(dfg_nodes, self.tokenizer, device)
        cdg_x = self.cdg_builder.nodes_to_embeddings(cdg_nodes, self.tokenizer, device)
        
        if len(icfg_edges) == 0:
            icfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            icfg_edge_index = torch.tensor(icfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(dfg_edges) == 0:
            dfg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            dfg_edge_index = torch.tensor(dfg_edges, dtype=torch.long, device=device).t().contiguous()
        
        if len(cdg_edges) == 0:
            cdg_edge_index = torch.tensor([[0], [0]], dtype=torch.long, device=device)
        else:
            cdg_edge_index = torch.tensor(cdg_edges, dtype=torch.long, device=device).t().contiguous()
        
        icfg_data = Data(x=icfg_x, edge_index=icfg_edge_index)
        dfg_data = Data(x=dfg_x, edge_index=dfg_edge_index)
        cdg_data = Data(x=cdg_x, edge_index=cdg_edge_index)
        
        return icfg_data, dfg_data, cdg_data
        
    def forward(self, code):
        device = next(self.parameters()).device
        icfg_data, dfg_data, cdg_data = self.build_graph_data(code, device)
        graph_repr = self.graph_fusion(icfg_data, dfg_data, cdg_data)
        
        tokens = self.tokenizer(
            code, 
            return_tensors='pt', 
            truncation=True, 
            max_length=512, 
            padding='max_length'
        )
        tokens = {k: v.to(device) for k, v in tokens.items()}
        
        code_output = self.graphcodebert(**tokens)
        code_repr = code_output.last_hidden_state[:, 0, :]
        code_repr = self.code_projection(code_repr)
        
        combined = torch.cat([graph_repr.unsqueeze(0), code_repr], dim=-1)
        fused_repr = self.multimodal_fusion(combined)
        
        return fused_repr.squeeze(0)


def generate_embeddings(model, input_csv, output_csv, device='mps'):
    df = pd.read_csv(input_csv)
    print(f'Loaded dataset: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    
    embeddings_list = []
    labels_list = []
    
    model.eval()
    with torch.no_grad():
        for idx in tqdm(range(len(df)), desc='Generating embeddings'):
            row = df.iloc[idx]
            code = str(row['func'])
            label = int(row['label'])
            
            embedding = model(code)
            embeddings_list.append(embedding.cpu().numpy())
            labels_list.append(label)
    
    embeddings_array = np.array(embeddings_list)
    
    embedding_columns = [f'emb_{i}' for i in range(embeddings_array.shape[1])]
    embeddings_df = pd.DataFrame(embeddings_array, columns=embedding_columns)
    embeddings_df['label'] = labels_list
    
    embeddings_df.to_csv(output_csv, index=False)
    print(f'\nEmbeddings saved to: {output_csv}')
    print(f'Output shape: {embeddings_df.shape}')
    print(f'Embedding dimension: {embeddings_array.shape[1]}')


if __name__ == '__main__':
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f'Device: {device}\n')
    
    model = EmbeddingExtractor(num_classes=6).to(device) 
    
    print('Loading trained model weights...')
    state_dict = torch.load('Download saved model from github shared location', map_location=device, weights_only=True)
    
    model_state_dict = model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items() if k in model_state_dict}
    model.load_state_dict(filtered_state_dict, strict=False)
    
    print(f'Loaded {len(filtered_state_dict)}/{len(state_dict)} parameters\n')
    
    input_csv = 'Your required dataset'
    output_csv = 'as per your specific location'
    
    generate_embeddings(model, input_csv, output_csv, device=device)
    
    print('\n✓ Done!')

# generated embedded dataset's assesment on different classifer overview

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error, roc_auc_score
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.regularizers import l1_l2
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.alpha'] = 0.3


class MetricsHistory(Callback):
    def __init__(self, X_val, y_val, X_test, y_test, num_classes):
        super().__init__()
        self.X_val = X_val
        self.y_val = y_val
        self.X_test = X_test
        self.y_test = y_test
        self.num_classes = num_classes
        self.history = {
            'val_loss': [], 'test_loss': [],
            'val_accuracy': [], 'test_accuracy': [],
            'val_precision': [], 'test_precision': [],
            'val_recall': [], 'test_recall': [],
            'val_f1': [], 'test_f1': [],
            'val_mcc': [], 'test_mcc': []
        }
    
    def on_epoch_end(self, epoch, logs=None):
        val_pred_proba = self.model.predict(self.X_val, verbose=0)
        val_pred = np.argmax(val_pred_proba, axis=-1) if val_pred_proba.ndim > 1 else val_pred_proba
        
        test_pred_proba = self.model.predict(self.X_test, verbose=0)
        test_pred = np.argmax(test_pred_proba, axis=-1) if test_pred_proba.ndim > 1 else test_pred_proba
        
        self.history['val_loss'].append(logs.get('val_loss', 0))
        self.history['test_loss'].append(logs.get('loss', 0))
        
        self.history['val_accuracy'].append(accuracy_score(self.y_val, val_pred))
        self.history['test_accuracy'].append(accuracy_score(self.y_test, test_pred))
        
        self.history['val_precision'].append(precision_score(self.y_val, val_pred, average='weighted', zero_division=0))
        self.history['test_precision'].append(precision_score(self.y_test, test_pred, average='weighted', zero_division=0))
        
        self.history['val_recall'].append(recall_score(self.y_val, val_pred, average='weighted', zero_division=0))
        self.history['test_recall'].append(recall_score(self.y_test, test_pred, average='weighted', zero_division=0))
        
        self.history['val_f1'].append(f1_score(self.y_val, val_pred, average='weighted', zero_division=0))
        self.history['test_f1'].append(f1_score(self.y_test, test_pred, average='weighted', zero_division=0))
        
        self.history['val_mcc'].append(matthews_corrcoef(self.y_val, val_pred))
        self.history['test_mcc'].append(matthews_corrcoef(self.y_test, test_pred))


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_label0 = train_df[train_df['label'] == 0].sample(n=3800, random_state=42)
    train_others = train_df[train_df['label'] != 0]
    train_df = pd.concat([train_label0, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    test_label0 = test_df[test_df['label'] == 0].sample(n=900, random_state=42)
    test_others = test_df[test_df['label'] != 0]
    test_df = pd.concat([test_label0, test_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_train_full = train_df.drop('label', axis=1).values
    y_train_full = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    return X_train, y_train, X_val, y_val, X_test, y_test


def train_gru_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    X_train_gru = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_val_gru = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
    X_test_gru = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
    
    model = Sequential()
    model.add(GRU(128, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True, 
                  kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4), recurrent_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    model.add(Dropout(0.6))
    model.add(BatchNormalization())
    model.add(GRU(64, kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4), recurrent_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    model.add(Dropout(0.6))
    model.add(BatchNormalization())
    model.add(Dense(num_classes, activation='softmax', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    history_callback = MetricsHistory(X_val_gru, y_val, X_test_gru, y_test, num_classes)
    model.fit(X_train_gru, y_train, epochs=epochs, batch_size=32, 
              validation_data=(X_val_gru, y_val), callbacks=[history_callback], verbose=0)
    
    return history_callback.history


def train_lstm_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train.shape[1],),
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.6),
        BatchNormalization(),
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.6),
        BatchNormalization(),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    history_callback = MetricsHistory(X_val, y_val, X_test, y_test, num_classes)
    model.fit(X_train, y_train_cat, epochs=epochs, batch_size=32,
             validation_data=(X_val, y_val_cat), callbacks=[history_callback], verbose=0)
    
    return history_callback.history


def train_deeptree_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=20, min_samples_split=10, 
                                          min_samples_leaf=5, ccp_alpha=0.001)
    dt_classifier.fit(X_train, y_train)
    
    X_train_transformed = dt_classifier.predict_proba(X_train)
    X_val_transformed = dt_classifier.predict_proba(X_val)
    X_test_transformed = dt_classifier.predict_proba(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_transformed.shape[1],), 
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.5),
        BatchNormalization(),
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.5),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.4),
        Dense(32, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.3),
        Dense(16, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    history_callback = MetricsHistory(X_val_transformed, y_val, X_test_transformed, y_test, num_classes)
    model.fit(X_train_transformed, y_train, epochs=epochs, batch_size=16, 
              validation_data=(X_val_transformed, y_val), callbacks=[history_callback], verbose=0)
    
    return history_callback.history


def train_xgboost_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    for n_est in range(10, 10 + epochs * 10, 10):
        model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42, n_estimators=n_est,
                                 max_depth=6, min_child_weight=3, gamma=0.1, 
                                 subsample=0.8, colsample_bytree=0.8, 
                                 reg_alpha=0.1, reg_lambda=1.0, learning_rate=0.05)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        
        val_pred_proba = model.predict_proba(X_val)
        test_pred_proba = model.predict_proba(X_test)
        
        from sklearn.metrics import log_loss
        history['val_loss'].append(log_loss(y_val, val_pred_proba))
        history['test_loss'].append(log_loss(y_test, test_pred_proba))
        
        history['val_accuracy'].append(accuracy_score(y_val, val_pred))
        history['test_accuracy'].append(accuracy_score(y_test, test_pred))
        
        history['val_precision'].append(precision_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(y_val, val_pred))
        history['test_mcc'].append(matthews_corrcoef(y_test, test_pred))
    
    return history


def train_svm_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    C_values = np.logspace(-2, 1, epochs)
    
    for C in C_values:
        model = SVC(kernel='linear', probability=True, random_state=42, C=C)
        model.fit(X_train, y_train)
        
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        
        val_pred_proba = model.predict_proba(X_val)
        test_pred_proba = model.predict_proba(X_test)
        
        from sklearn.metrics import log_loss
        history['val_loss'].append(log_loss(y_val, val_pred_proba))
        history['test_loss'].append(log_loss(y_test, test_pred_proba))
        
        history['val_accuracy'].append(accuracy_score(y_val, val_pred))
        history['test_accuracy'].append(accuracy_score(y_test, test_pred))
        
        history['val_precision'].append(precision_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(y_val, val_pred))
        history['test_mcc'].append(matthews_corrcoef(y_test, test_pred))
    
    return history


def train_sgd_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    alpha_values = np.logspace(-4, -1, epochs)
    
    for alpha in alpha_values:
        model = SGDClassifier(loss='hinge', penalty='elasticnet', alpha=alpha, l1_ratio=0.15,
                             learning_rate='optimal', max_iter=1000, early_stopping=False,
                             random_state=42)
        model.fit(X_train, y_train)
        
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        
        val_decision = model.decision_function(X_val)
        test_decision = model.decision_function(X_test)
        
        history['val_loss'].append(np.mean(np.maximum(0, 1 - y_val * val_decision.mean(axis=1 if val_decision.ndim > 1 else 0))))
        history['test_loss'].append(np.mean(np.maximum(0, 1 - y_test * test_decision.mean(axis=1 if test_decision.ndim > 1 else 0))))
        
        history['val_accuracy'].append(accuracy_score(y_val, val_pred))
        history['test_accuracy'].append(accuracy_score(y_test, test_pred))
        
        history['val_precision'].append(precision_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(y_val, val_pred))
        history['test_mcc'].append(matthews_corrcoef(y_test, test_pred))
    
    return history


def train_decision_tree_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    depths = range(5, 5 + epochs)
    
    for depth in depths:
        model = DecisionTreeClassifier(max_depth=depth, min_samples_split=10, min_samples_leaf=5, 
                                      ccp_alpha=0.001, random_state=42)
        model.fit(X_train, y_train)
        
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        
        val_pred_proba = model.predict_proba(X_val)
        test_pred_proba = model.predict_proba(X_test)
        
        from sklearn.metrics import log_loss
        history['val_loss'].append(log_loss(y_val, val_pred_proba))
        history['test_loss'].append(log_loss(y_test, test_pred_proba))
        
        history['val_accuracy'].append(accuracy_score(y_val, val_pred))
        history['test_accuracy'].append(accuracy_score(y_test, test_pred))
        
        history['val_precision'].append(precision_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(y_val, val_pred))
        history['test_mcc'].append(matthews_corrcoef(y_test, test_pred))
    
    return history


def train_random_forest_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    n_estimators_list = range(20, 20 + epochs * 10, 10)
    
    for n_est in n_estimators_list:
        model = RandomForestClassifier(n_estimators=n_est, min_samples_split=10, max_depth=15,
                                      max_features='sqrt', min_samples_leaf=4, ccp_alpha=0.001,
                                      random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)
        
        val_pred = model.predict(X_val)
        test_pred = model.predict(X_test)
        
        val_pred_proba = model.predict_proba(X_val)
        test_pred_proba = model.predict_proba(X_test)
        
        from sklearn.metrics import log_loss
        history['val_loss'].append(log_loss(y_val, val_pred_proba))
        history['test_loss'].append(log_loss(y_test, test_pred_proba))
        
        history['val_accuracy'].append(accuracy_score(y_val, val_pred))
        history['test_accuracy'].append(accuracy_score(y_test, test_pred))
        
        history['val_precision'].append(precision_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(y_val, val_pred, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(y_test, test_pred, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(y_val, val_pred))
        history['test_mcc'].append(matthews_corrcoef(y_test, test_pred))
    
    return history


def train_single_layer_nn_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    model = Sequential([
        Dense(num_classes, activation='softmax', input_shape=(X_train.shape[1],),
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4))
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    history_callback = MetricsHistory(X_val, y_val, X_test, y_test, num_classes)
    model.fit(X_train, y_train_cat, epochs=epochs, batch_size=32,
             validation_data=(X_val, y_val_cat), callbacks=[history_callback], verbose=0)
    
    return history_callback.history


class NumericalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {'features': self.features[idx], 'labels': self.labels[idx]}


class NumericalTransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(NumericalTransformerClassifier, self).__init__()
        self.transformer = BertModel.from_pretrained('bert-base-uncased')
        self.embedding = nn.Linear(input_dim, self.transformer.config.hidden_size)
        self.dropout = nn.Dropout(0.6)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, num_labels)

    def forward(self, features):
        embeddings = self.embedding(features)
        transformer_output = self.transformer(inputs_embeds=embeddings.unsqueeze(1)).last_hidden_state[:, 0, :]
        output = self.classifier(self.dropout(transformer_output))
        return output


def train_bert_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs=20):
    train_dataset = NumericalDataset(X_train, y_train)
    val_dataset = NumericalDataset(X_val, y_val)
    test_dataset = NumericalDataset(X_test, y_test)
    
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = NumericalTransformerClassifier(input_dim, num_classes)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-6, weight_decay=0.01)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    history = {
        'val_loss': [], 'test_loss': [],
        'val_accuracy': [], 'test_accuracy': [],
        'val_precision': [], 'test_precision': [],
        'val_recall': [], 'test_recall': [],
        'val_f1': [], 'test_f1': [],
        'val_mcc': [], 'test_mcc': []
    }
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0
        test_loss = 0
        val_preds = []
        test_preds = []
        val_trues = []
        test_trues = []
        
        with torch.no_grad():
            for batch in val_loader:
                features = batch['features'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                _, preds = torch.max(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_trues.extend(labels.cpu().numpy())
            
            for batch in test_loader:
                features = batch['features'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(features)
                test_loss += criterion(outputs, labels).item()
                _, preds = torch.max(outputs, dim=1)
                test_preds.extend(preds.cpu().numpy())
                test_trues.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        test_loss /= len(test_loader)
        
        val_preds = np.array(val_preds)
        test_preds = np.array(test_preds)
        val_trues = np.array(val_trues)
        test_trues = np.array(test_trues)
        
        history['val_loss'].append(val_loss)
        history['test_loss'].append(test_loss)
        
        history['val_accuracy'].append(accuracy_score(val_trues, val_preds))
        history['test_accuracy'].append(accuracy_score(test_trues, test_preds))
        
        history['val_precision'].append(precision_score(val_trues, val_preds, average='weighted', zero_division=0))
        history['test_precision'].append(precision_score(test_trues, test_preds, average='weighted', zero_division=0))
        
        history['val_recall'].append(recall_score(val_trues, val_preds, average='weighted', zero_division=0))
        history['test_recall'].append(recall_score(test_trues, test_preds, average='weighted', zero_division=0))
        
        history['val_f1'].append(f1_score(val_trues, val_preds, average='weighted', zero_division=0))
        history['test_f1'].append(f1_score(test_trues, test_preds, average='weighted', zero_division=0))
        
        history['val_mcc'].append(matthews_corrcoef(val_trues, val_preds))
        history['test_mcc'].append(matthews_corrcoef(test_trues, test_preds))
    
    return history


def create_figure1_loss_comparison(all_histories, all_model_names):
    n_models = len(all_model_names)
    n_cols = 4
    n_rows = (n_models + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    axes = axes.flatten() if n_models > 1 else [axes]
    
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for idx, (model_name, history) in enumerate(zip(all_model_names, all_histories)):
        ax = axes[idx]
        
        epochs_range = range(1, len(history['val_loss']) + 1)
        
        ax.plot(epochs_range, history['val_loss'], 
                color=colors[idx % 10], linewidth=2.5, marker='o', markersize=6, 
                label='Validation Loss', alpha=0.9)
        ax.plot(epochs_range, history['test_loss'], 
                color=colors[idx % 10], linewidth=2.5, marker='s', markersize=6, 
                linestyle='--', label='Test Loss', alpha=0.7)
        
        ax.set_xlabel('Iteration', fontsize=12, fontweight='bold')
        ax.set_ylabel('Loss', fontsize=12, fontweight='bold')
        ax.set_title(f'{model_name}', fontsize=14, fontweight='bold', pad=10)
        ax.legend(loc='upper right', frameon=True, shadow=True, fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle('Validation vs Test Loss Convergence Analysis', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.savefig('figure1_loss_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig('figure1_loss_comparison.pdf', bbox_inches='tight', facecolor='white')
    print("Figure 1 saved: Loss comparison for all classifiers")


def create_figure2_metrics_evolution(all_histories, all_model_names):
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'mcc']
    metric_titles = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'MCC']
    
    fig, axes = plt.subplots(2, 3, figsize=(24, 12))
    axes = axes.flatten()
    
    colors = plt.cm.tab10(np.linspace(0, 1, 10))
    
    for metric_idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
        ax = axes[metric_idx]
        
        for model_idx, (model_name, history) in enumerate(zip(all_model_names, all_histories)):
            epochs_range = range(1, len(history[f'val_{metric}']) + 1)
            
            ax.plot(epochs_range, history[f'val_{metric}'], 
                   color=colors[model_idx % 10], linewidth=2, marker='o', 
                   markersize=4, label=f'{model_name} (Val)', alpha=0.8)
            ax.plot(epochs_range, history[f'test_{metric}'], 
                   color=colors[model_idx % 10], linewidth=2, marker='s', 
                   markersize=4, linestyle='--', label=f'{model_name} (Test)', alpha=0.6)
        
        ax.set_xlabel('Iteration', fontsize=11, fontweight='bold')
        ax.set_ylabel(title, fontsize=11, fontweight='bold')
        ax.set_title(f'{title} Evolution', fontsize=13, fontweight='bold', pad=8)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True, fontsize=7)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    axes[5].axis('off')
    
    fig.suptitle('Comprehensive Multi-Class Classification Metrics Evolution', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.savefig('figure2_metrics_evolution.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig('figure2_metrics_evolution.pdf', bbox_inches='tight', facecolor='white')
    print("Figure 2 saved: Metrics evolution for all classifiers")


def create_figure3_learning_trajectories(all_histories, all_model_names):
    n_models = len(all_model_names)
    groups = []
    
    for i in range(0, n_models, 3):
        groups.append((all_model_names[i:i+3], all_histories[i:i+3]))
    
    n_groups = len(groups)
    n_cols = 3
    n_rows = (n_groups + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 8 * n_rows))
    if n_groups > 1:
        axes = axes.flatten()
    else:
        axes = [axes] if n_groups == 1 else axes.flatten()
    
    colors_set = ['#2E86AB', '#A23B72', '#F18F01']
    
    for group_idx, (names, histories) in enumerate(groups):
        ax = axes[group_idx]
        
        for model_idx, (model_name, history) in enumerate(zip(names, histories)):
            epochs_range = range(1, len(history['val_f1']) + 1)
            
            ax.plot(epochs_range, history['val_f1'], 
                    color=colors_set[model_idx % 3], linewidth=3, marker='o', 
                    markersize=7, label=f'{model_name} Validation', alpha=0.8)
            ax.plot(epochs_range, history['test_f1'], 
                    color=colors_set[model_idx % 3], linewidth=3, marker='s', 
                    markersize=7, linestyle='--', label=f'{model_name} Test', alpha=0.6)
        
        ax.set_xlabel('Iteration', fontsize=13, fontweight='bold')
        ax.set_ylabel('F1-Score', fontsize=13, fontweight='bold')
        ax.set_title(f'Model Learning Trajectories (Group {group_idx + 1})', fontsize=15, fontweight='bold', pad=10)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True, fontsize=10)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    for idx in range(n_groups, len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle('Model Learning Trajectories - F1 Score Evolution', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.savefig('figure3_learning_trajectories.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig('figure3_learning_trajectories.pdf', bbox_inches='tight', facecolor='white')
    print("Figure 3 saved: Learning trajectories for all classifiers")


def create_figure4_convergence_analysis(all_histories, all_model_names):
    n_models = len(all_model_names)
    groups = []
    
    for i in range(0, n_models, 3):
        groups.append((all_model_names[i:i+3], all_histories[i:i+3]))
    
    n_groups = len(groups)
    n_cols = 3
    n_rows = (n_groups + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 8 * n_rows))
    if n_groups > 1:
        axes = axes.flatten()
    else:
        axes = [axes] if n_groups == 1 else axes.flatten()
    
    colors_set = ['#2E86AB', '#A23B72', '#F18F01']
    
    for group_idx, (names, histories) in enumerate(groups):
        ax = axes[group_idx]
        
        for model_idx, (model_name, history) in enumerate(zip(names, histories)):
            val_loss = np.array(history['val_loss'])
            convergence_rate = np.abs(np.diff(val_loss))
            epochs_range = range(1, len(convergence_rate) + 1)
            
            ax.plot(epochs_range, convergence_rate, 
                   color=colors_set[model_idx % 3], linewidth=3, 
                   marker='D', markersize=6, label=model_name, alpha=0.8)
        
        ax.set_xlabel('Iteration', fontsize=13, fontweight='bold')
        ax.set_ylabel('|ΔLoss|', fontsize=13, fontweight='bold')
        ax.set_title(f'Convergence Rate Analysis (Group {group_idx + 1})', fontsize=15, fontweight='bold', pad=10)
        ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True, shadow=True, fontsize=11)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
    
    for idx in range(n_groups, len(axes)):
        axes[idx].axis('off')
    
    fig.suptitle('Convergence Rate Analysis - Validation Loss Gradient', 
                 fontsize=18, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.savefig('figure4_convergence_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig('figure4_convergence_analysis.pdf', bbox_inches='tight', facecolor='white')
    print("Figure 4 saved: Convergence rate analysis for all classifiers")


if __name__ == '__main__':
    train_path = 'Train dataset path'
    test_path = 'test dtaaset path'
    
    print('Loading and sampling data...')
    X_train, y_train, X_val, y_val, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Validation samples: {X_val.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    epochs = 20
    
    print('\n' + '='*80)
    print('Training all classifiers and collecting metrics over iterations...')
    print('='*80)
    
    all_histories = []
    all_model_names = []
    
    print('\n[1/10] Training Single Layer NN...')
    history = train_single_layer_nn_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('Single Layer NN')
    
    print('[2/10] Training GRU...')
    history = train_gru_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('GRU')
    
    print('[3/10] Training LSTM...')
    history = train_lstm_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('LSTM')
    
    print('[4/10] Training DeepTree...')
    history = train_deeptree_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('DeepTree')
    
    print('[5/10] Training XGBoost...')
    history = train_xgboost_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('XGBoost')
    
    print('[6/10] Training SVM...')
    history = train_svm_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('SVM')
    
    print('[7/10] Training SGD...')
    history = train_sgd_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('SGD')
    
    print('[8/10] Training Decision Tree...')
    history = train_decision_tree_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('Decision Tree')
    
    print('[9/10] Training Random Forest...')
    history = train_random_forest_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('Random Forest')
    
    print('[10/10] Training BERT...')
    history = train_bert_with_history(X_train, y_train, X_val, y_val, X_test, y_test, num_classes, epochs)
    all_histories.append(history)
    all_model_names.append('BERT')
    
    print('\n' + '='*80)
    print('Generating publication-quality figures...')
    print('='*80)
    
    print('\nCreating Figure 1: Validation vs Test Loss for all classifiers...')
    create_figure1_loss_comparison(all_histories, all_model_names)
    
    print('\nCreating Figure 2: Comprehensive metrics evolution for all classifiers...')
    create_figure2_metrics_evolution(all_histories, all_model_names)
    
    print('\nCreating Figure 3: Model learning trajectories (3 models per figure)...')
    create_figure3_learning_trajectories(all_histories, all_model_names)
    
    print('\nCreating Figure 4: Convergence rate analysis (3 models per figure)...')
    create_figure4_convergence_analysis(all_histories, all_model_names)
    


## train vs Val vs Test overfitting analysis of generted embed dataset

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, AdamW
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score,
    matthews_corrcoef, cohen_kappa_score, mean_squared_error, mean_absolute_error
)
import xgboost as xgb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l1_l2
import tensorflow as tf
import warnings
warnings.filterwarnings('ignore')


def load_and_sample_data(train_path, test_path):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    train_label0 = train_df[train_df['label'] == 0].sample(n=3800, random_state=42)
    train_others = train_df[train_df['label'] != 0]
    train_df = pd.concat([train_label0, train_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    test_label0 = test_df[test_df['label'] == 0].sample(n=900, random_state=42)
    test_others = test_df[test_df['label'] != 0]
    test_df = pd.concat([test_label0, test_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_train_full = train_df.drop('label', axis=1).values
    y_train_full = train_df['label'].values
    X_test = test_df.drop('label', axis=1).values
    y_test = test_df['label'].values
    
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
    )
    
    return X_train, y_train, X_val, y_val, X_test, y_test


def calculate_metrics(y_true, y_pred, y_pred_proba=None, num_classes=6):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    kappa = cohen_kappa_score(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    cm = confusion_matrix(y_true, y_pred)
    tp = np.diag(cm).sum()
    fn = cm.sum(axis=1).sum() - tp
    
    auc = 0.0
    if y_pred_proba is not None:
        try:
            auc = roc_auc_score(pd.get_dummies(y_true), y_pred_proba, multi_class='ovr', average='macro')
        except:
            auc = 0.0
    
    return {
        'AUC': auc,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'MCC': mcc,
        'Kappa': kappa,
        'MSE': mse,
        'MAE': mae,
        'TP': tp,
        'FN': fn
    }


def print_results(model_name, train_metrics, val_metrics, test_metrics):
    print(f'\n{"="*120}')
    print(f'{model_name:^120}')
    print(f'{"="*120}')
    
    print(f'{"Metric":<15} {"Training":<20} {"Validation":<20} {"Test":<20}')
    print(f'{"-"*120}')
    
    metrics_names = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1', 'MCC', 'Kappa', 'MSE', 'MAE', 'TP', 'FN']
    
    for metric in metrics_names:
        train_val = train_metrics[metric]
        val_val = val_metrics[metric]
        test_val = test_metrics[metric]
        
        if isinstance(train_val, (int, np.integer)):
            print(f'{metric:<15} {train_val:<20} {val_val:<20} {test_val:<20}')
        else:
            print(f'{metric:<15} {train_val:<20.4f} {val_val:<20.4f} {test_val:<20.4f}')
    
    print(f'{"="*120}\n')


def train_gru(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training GRU...')
    
    X_train_gru = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
    X_val_gru = X_val.reshape((X_val.shape[0], 1, X_val.shape[1]))
    X_test_gru = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))
    
    model = Sequential()
    model.add(GRU(128, input_shape=(X_train_gru.shape[1], X_train_gru.shape[2]), return_sequences=True, 
                  kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4), recurrent_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    model.add(Dropout(0.6))
    model.add(BatchNormalization())
    model.add(GRU(64, kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4), recurrent_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    model.add(Dropout(0.6))
    model.add(BatchNormalization())
    model.add(Dense(num_classes, activation='softmax', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
    
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(X_train_gru, y_train, epochs=50, batch_size=32, 
              validation_data=(X_val_gru, y_val), callbacks=[early_stopping], verbose=0)
    
    y_train_pred_proba = model.predict(X_train_gru, verbose=0)
    y_train_pred = np.argmax(y_train_pred_proba, axis=-1)
    
    y_val_pred_proba = model.predict(X_val_gru, verbose=0)
    y_val_pred = np.argmax(y_val_pred_proba, axis=-1)
    
    y_test_pred_proba = model.predict(X_test_gru, verbose=0)
    y_test_pred = np.argmax(y_test_pred_proba, axis=-1)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('GRU MODEL', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training XGBoost...')
    model = xgb.XGBClassifier(eval_metric='mlogloss', random_state=42, 
                             max_depth=6, min_child_weight=3, gamma=0.1, 
                             subsample=0.8, colsample_bytree=0.8, 
                             reg_alpha=0.1, reg_lambda=1.0, learning_rate=0.05)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    
    y_train_pred = model.predict(X_train)
    y_train_pred_proba = model.predict_proba(X_train)
    
    y_val_pred = model.predict(X_val)
    y_val_pred_proba = model.predict_proba(X_val)
    
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('XGBOOST MODEL', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_svm(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training SVM...')
    model = SVC(kernel='linear', probability=True, random_state=42, C=0.1)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_train_pred_proba = model.predict_proba(X_train)
    
    y_val_pred = model.predict(X_val)
    y_val_pred_proba = model.predict_proba(X_val)
    
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('SVM MODEL', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_deeptree(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training DeepTree...')
    dt_classifier = DecisionTreeClassifier(random_state=42, max_depth=20, min_samples_split=10, 
                                          min_samples_leaf=5, ccp_alpha=0.001)
    dt_classifier.fit(X_train, y_train)
    
    X_train_transformed = dt_classifier.predict_proba(X_train)
    X_val_transformed = dt_classifier.predict_proba(X_val)
    X_test_transformed = dt_classifier.predict_proba(X_test)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train_transformed.shape[1],), 
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.5),
        BatchNormalization(),
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.5),
        BatchNormalization(),
        Dense(64, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.4),
        Dense(32, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.3),
        Dense(16, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    model.fit(X_train_transformed, y_train, epochs=50, batch_size=16, 
              validation_data=(X_val_transformed, y_val), callbacks=[early_stopping], verbose=0)
    
    y_train_pred_proba = model.predict(X_train_transformed, verbose=0)
    y_train_pred = np.argmax(y_train_pred_proba, axis=1)
    
    y_val_pred_proba = model.predict(X_val_transformed, verbose=0)
    y_val_pred = np.argmax(y_val_pred_proba, axis=1)
    
    y_test_pred_proba = model.predict(X_test_transformed, verbose=0)
    y_test_pred = np.argmax(y_test_pred_proba, axis=1)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('DEEPTREE MODEL', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_sgd(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training SGD...')
    model = SGDClassifier(loss='hinge', penalty='elasticnet', alpha=0.001, l1_ratio=0.15,
                         learning_rate='optimal', max_iter=1000, early_stopping=True,
                         validation_fraction=0.2, n_iter_no_change=10, random_state=42)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, None, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, None, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, None, num_classes)
    
    print_results('SGD CLASSIFIER', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_decision_tree(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training Decision Tree...')
    model = DecisionTreeClassifier(max_depth=15, min_samples_split=10, min_samples_leaf=5, 
                                  ccp_alpha=0.001, random_state=42)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_train_pred_proba = model.predict_proba(X_train)
    
    y_val_pred = model.predict(X_val)
    y_val_pred_proba = model.predict_proba(X_val)
    
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('DECISION TREE', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training Random Forest...')
    model = RandomForestClassifier(n_estimators=100, min_samples_split=10, max_depth=15,
                                  max_features='sqrt', min_samples_leaf=4, ccp_alpha=0.001,
                                  random_state=42, n_jobs=-1)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_train_pred_proba = model.predict_proba(X_train)
    
    y_val_pred = model.predict(X_val)
    y_val_pred_proba = model.predict_proba(X_val)
    
    y_test_pred = model.predict(X_test)
    y_test_pred_proba = model.predict_proba(X_test)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('RANDOM FOREST', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_single_layer_nn(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training Single Layer NN...')
    
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    model = Sequential([
        Dense(num_classes, activation='softmax', input_shape=(X_train.shape[1],),
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4))
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    model.fit(X_train, y_train_cat, epochs=100, batch_size=32,
             validation_data=(X_val, y_val_cat), callbacks=[early_stopping], verbose=0)
    
    y_train_pred_proba = model.predict(X_train, verbose=0)
    y_train_pred = np.argmax(y_train_pred_proba, axis=1)
    
    y_val_pred_proba = model.predict(X_val, verbose=0)
    y_val_pred = np.argmax(y_val_pred_proba, axis=1)
    
    y_test_pred_proba = model.predict(X_test, verbose=0)
    y_test_pred = np.argmax(y_test_pred_proba, axis=1)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('SINGLE LAYER NN', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


def train_lstm(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training LSTM...')
    
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
    y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes)
    y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)
    
    model = Sequential([
        Dense(256, activation='relu', input_shape=(X_train.shape[1],),
              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.6),
        BatchNormalization(),
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)),
        Dropout(0.6),
        BatchNormalization(),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                 loss='categorical_crossentropy', metrics=['accuracy'])
    
    early_stopping = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    model.fit(X_train, y_train_cat, epochs=100, batch_size=32,
             validation_data=(X_val, y_val_cat), callbacks=[early_stopping], verbose=0)
    
    y_train_pred_proba = model.predict(X_train, verbose=0)
    y_train_pred = np.argmax(y_train_pred_proba, axis=1)
    
    y_val_pred_proba = model.predict(X_val, verbose=0)
    y_val_pred = np.argmax(y_val_pred_proba, axis=1)
    
    y_test_pred_proba = model.predict(X_test, verbose=0)
    y_test_pred = np.argmax(y_test_pred_proba, axis=1)
    
    train_metrics = calculate_metrics(y_train, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('LSTM MODEL', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


class NumericalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return {'features': self.features[idx], 'labels': self.labels[idx]}


class NumericalTransformerClassifier(nn.Module):
    def __init__(self, input_dim, num_labels):
        super(NumericalTransformerClassifier, self).__init__()
        self.transformer = BertModel.from_pretrained('bert-base-uncased')
        self.embedding = nn.Linear(input_dim, self.transformer.config.hidden_size)
        self.dropout = nn.Dropout(0.6)
        self.classifier = nn.Linear(self.transformer.config.hidden_size, num_labels)

    def forward(self, features):
        embeddings = self.embedding(features)
        transformer_output = self.transformer(inputs_embeds=embeddings.unsqueeze(1)).last_hidden_state[:, 0, :]
        output = self.classifier(self.dropout(transformer_output))
        return output


def evaluate_bert(model, data_loader, device):
    model.eval()
    predictions, actuals, all_outputs_prob = [], [], []
    with torch.no_grad():
        for batch in data_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(features)
            
            outputs_prob = nn.functional.softmax(outputs, dim=1).cpu().numpy()
            all_outputs_prob.extend(outputs_prob)
            
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    
    return np.array(actuals), np.array(predictions), np.array(all_outputs_prob)


def train_bert(X_train, y_train, X_val, y_val, X_test, y_test, num_classes):
    print('Training BERT...')
    train_dataset = NumericalDataset(X_train, y_train)
    val_dataset = NumericalDataset(X_val, y_val)
    test_dataset = NumericalDataset(X_test, y_test)
    
    batch_size = 64
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    input_dim = X_train.shape[1]
    model = NumericalTransformerClassifier(input_dim, num_classes)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=2e-6, weight_decay=0.01)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    
    best_val_loss = float('inf')
    patience_counter = 0
    patience = 10
    
    epochs = 50
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            features = batch['features'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                features = batch['features'].to(device)
                labels = batch['labels'].to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
        
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                model.load_state_dict(best_model_state)
                break
    
    y_train_true, y_train_pred, y_train_pred_proba = evaluate_bert(model, train_loader, device)
    y_val_true, y_val_pred, y_val_pred_proba = evaluate_bert(model, val_loader, device)
    y_test_true, y_test_pred, y_test_pred_proba = evaluate_bert(model, test_loader, device)
    
    train_metrics = calculate_metrics(y_train_true, y_train_pred, y_train_pred_proba, num_classes)
    val_metrics = calculate_metrics(y_val_true, y_val_pred, y_val_pred_proba, num_classes)
    test_metrics = calculate_metrics(y_test_true, y_test_pred, y_test_pred_proba, num_classes)
    
    print_results('BERT TRANSFORMER', train_metrics, val_metrics, test_metrics)
    return train_metrics, val_metrics, test_metrics


if __name__ == '__main__':
    train_path = 'Train dataset path'
    test_path = 'test dtaaset path'
    
    print('Loading and sampling data...')
    X_train, y_train, X_val, y_val, X_test, y_test = load_and_sample_data(train_path, test_path)
    
    num_classes = len(np.unique(y_train))
    
    print(f'\nDataset Information:')
    print(f'  Training samples:   {X_train.shape[0]}')
    print(f'  Validation samples: {X_val.shape[0]}')
    print(f'  Test samples:       {X_test.shape[0]}')
    print(f'  Feature dimensions: {X_train.shape[1]}')
    print(f'  Number of classes:  {num_classes}')
    
    print(f'\nTraining Label Distribution:')
    unique, counts = np.unique(y_train, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nValidation Label Distribution:')
    unique, counts = np.unique(y_val, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    print(f'\nTest Label Distribution:')
    unique, counts = np.unique(y_test, return_counts=True)
    for label, count in zip(unique, counts):
        print(f'  Class {label}: {count}')
    
    results = {}
    
    train_m, val_m, test_m = train_single_layer_nn(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['Single Layer NN'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_gru(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['GRU'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_xgboost(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['XGBoost'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_svm(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['SVM'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_deeptree(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['DeepTree'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_sgd(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['SGD'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_decision_tree(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['Decision Tree'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_random_forest(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['Random Forest'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_lstm(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['LSTM'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    train_m, val_m, test_m = train_bert(X_train, y_train, X_val, y_val, X_test, y_test, num_classes)
    results['BERT'] = {'train': train_m, 'val': val_m, 'test': test_m}
    
    print('\n' + '='*150)
    print(f'{"SUMMARY - TEST SET RESULTS":^150}')
    print('='*150)
    print(f'{"Model":<18} {"AUC":<10} {"Acc":<10} {"Pre":<10} {"Rec":<10} {"F1":<10} {"MCC":<10} {"Kappa":<10} {"MSE":<10} {"MAE":<10} {"TP":<8} {"FN":<8}')
    print('-'*150)
    
    for model_name, metrics_dict in results.items():
        test_metrics = metrics_dict['test']
        print(f'{model_name:<18} {test_metrics["AUC"]:<10.4f} {test_metrics["Accuracy"]:<10.4f} {test_metrics["Precision"]:<10.4f} '
              f'{test_metrics["Recall"]:<10.4f} {test_metrics["F1"]:<10.4f} {test_metrics["MCC"]:<10.4f} {test_metrics["Kappa"]:<10.4f} '
              f'{test_metrics["MSE"]:<10.4f} {test_metrics["MAE"]:<10.4f} {test_metrics["TP"]:<8} {test_metrics["FN"]:<8}')
    
    print('='*150)
    
    best_model = max(results.items(), key=lambda x: x[1]['test']['F1'])
    print(f'\nBest Model (Test F1): {best_model[0]} (F1 Score: {best_model[1]["test"]["F1"]:.4f})')
    
    print('\n' + '='*150)
    print(f'{"OVERFITTING ANALYSIS (Training vs Test)":^150}')
    print('='*150)
    print(f'{"Model":<18} {"Train F1":<12} {"Test F1":<12} {"Gap":<12} {"Status":<20}')
    print('-'*150)


## 1 layer NN classifer seperate execution

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, classification_report, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score, roc_auc_score,
                             log_loss, balanced_accuracy_score, hamming_loss)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

train_path = '/Users/akter/fahim/codegraph/traintry1.csv'
test_path = '/Users/akter/fahim/codegraph/testtry1.csv'

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)
test_label0 = test_data[test_data['label'] == 0].sample(n=900, random_state=42)
test_others = test_data[test_data['label'] != 0]
test_data = pd.concat([test_label0, test_others], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)


X_train = train_data.iloc[:, :-1].values
y_train = train_data.iloc[:, -1].values

X_test = test_data.iloc[:, :-1].values
y_test = test_data.iloc[:, -1].values

model = MLPClassifier(hidden_layer_sizes=(), activation='logistic', 
                      solver='adam', max_iter=1000, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

print("="*70)
print("CLASSIFICATION METRICS RESULTS")
print("="*70)

accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy: {accuracy:.4f}")

balanced_acc = balanced_accuracy_score(y_test, y_pred)
print(f"Balanced Accuracy: {balanced_acc:.4f}")

precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
precision_micro = precision_score(y_test, y_pred, average='micro', zero_division=0)
precision_weighted = precision_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"\nPrecision (Macro): {precision_macro:.4f}")
print(f"Precision (Micro): {precision_micro:.4f}")
print(f"Precision (Weighted): {precision_weighted:.4f}")

recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
recall_micro = recall_score(y_test, y_pred, average='micro', zero_division=0)
recall_weighted = recall_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"\nRecall (Macro): {recall_macro:.4f}")
print(f"Recall (Micro): {recall_micro:.4f}")
print(f"Recall (Weighted): {recall_weighted:.4f}")

f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1_micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"\nF1-Score (Macro): {f1_macro:.4f}")
print(f"F1-Score (Micro): {f1_micro:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")

mcc = matthews_corrcoef(y_test, y_pred)
print(f"\nMatthews Correlation Coefficient: {mcc:.4f}")

kappa = cohen_kappa_score(y_test, y_pred)
print(f"Cohen's Kappa: {kappa:.4f}")

hamming = hamming_loss(y_test, y_pred)
print(f"Hamming Loss: {hamming:.4f}")

logloss = log_loss(y_test, y_pred_proba)
print(f"Log Loss: {logloss:.4f}")

classes = np.unique(y_train)
y_test_binarized = label_binarize(y_test, classes=classes)
if len(classes) == 2:
    roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])
else:
    roc_auc_ovr = roc_auc_score(y_test_binarized, y_pred_proba, 
                                 average='macro', multi_class='ovr')
    roc_auc_ovo = roc_auc_score(y_test_binarized, y_pred_proba, 
                                 average='macro', multi_class='ovo')

print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

print("\n" + "="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, zero_division=0))

